## Acknowledgements

To start, I’d like to thank **Tong Hui Kang** and **konbu17**.

The CoT data and part of the hyperparameter settings used in this notebook are adapted from Tong Hui Kang’s open-source GitHub repository, while the training code is modified based on konbu17’s public notebook.  
- [Tong Hui Kang’s open-source GitHub repository](https://github.com/tonghuikang/nemotron)
- [konbu17’s public notebook](https://www.kaggle.com/code/konbu17/nemotron-sft-lora-with-cot)

## Overview

In this notebook, I will try to reproduce the method released by Tong Hui Kang and train a model that can reach around **0.85** on the public leaderboard.

However, due to differences in the training platform, I can only make my setup as close as possible to Tong Hui Kang’s original configuration. In addition, I did not use the generated augmentation data in this reproduction. The scripts for generating such augmented data can be found in Tong Hui Kang’s repository.

As a result, the final score of this notebook is expected to be slightly lower than the score of Tong Hui Kang’s currently public model.

## Notes on Training

Also, I am still a beginner in practical LLM training, so there are likely many inefficiencies in my training setup.

Although I tried to speed up the process with **Unsloth**, I only managed to reduce the training time from **10+ hours** to **7+ hours**. By comparison, Tong Hui Kang mentioned when sharing his method publicly that even without using **Tinker**, each of his training runs only took **4+ hours**.

Even so, being able to train a model like this by myself was still a rewarding experience. I hope this notebook can also help others participate more effectively in this competition.

In [ ]:

import os, sys
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="strict")

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
DATASET_PATH    = "/kaggle/input/datasets/llkh0a/nvidia-nemotron-distiled-dataset/train_split_with_cot_0.81.csv"
ADAPTER_DIR     = "/kaggle/working/sft_adapter"
OUTPUT_DIR      = "/kaggle/working"

# ── Sample budget per category (None = use all available) ──────────────────
TYPE_SAMPLES = {
    "bit_manipulation":        None,
    "cipher":                  None,
    "cryptarithm_deduce":      None,
    "cryptarithm_guess":       None,
    "equation_numeric_deduce": None,
    "equation_numeric_guess":  None,
    "unit_conversion":         None,
    "numeral":                 None,
    "gravity":                 None,
}

# ── Model ──────────────────────────────────────────────────────────────────
MAX_SEQ_LEN = 8192

# ── LoRA ───────────────────────────────────────────────────────────────────
LORA_RANK    = 32
LORA_ALPHA   = 32
LORA_DROPOUT = 0.0
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "in_proj", "out_proj", "up_proj", "down_proj",
    "lm_head",
]

# ── Training ───────────────────────────────────────────────────────────────
# Memory analysis (RTX Pro 6000, 94.97 GB):
#   Static (model+LoRA+optimizer): 62.15 GB
#   μ=1 dynamic: ~7.4 GB  → total ~70 GB  ✓
#   μ=2 dynamic: ~14.8 GB → total ~77 GB  ✓ (safe, ~2x faster)
#   μ=4 dynamic: ~29.6 GB → total ~92 GB+ → OOM ✗
SEED                   = 123
NUM_EPOCHS             = 1
BATCH_SIZE             = 2     # μ=2: estimated ~77 GB, fits within 94.97 GB
GRAD_ACCUM             = 32    # keeps effective batch = BATCH_SIZE * GRAD_ACCUM = 64
LR                     = 2e-4
LR_SCHEDULER           = "linear"
WARMUP_STEPS           = 0
WEIGHT_DECAY           = 0.0
MAX_GRAD_NORM          = 1e9
ADAM_BETA1             = 0.9
ADAM_BETA2             = 0.95
ADAM_EPSILON           = 1e-8
LOGGING_STEPS          = 10
DATALOADER_NUM_WORKERS = 2

PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

print("Config ready.")
print(f"  dataset    : {DATASET_PATH}")
print(f"  model      : {BASE_MODEL_NAME}")
print(f"  LoRA       : rank={LORA_RANK}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print(f"  microbatch : {BATCH_SIZE}  grad_accum={GRAD_ACCUM}  eff_batch={BATCH_SIZE*GRAD_ACCUM}")
print(f"  training   : epochs={NUM_EPOCHS}, lr={LR}, scheduler={LR_SCHEDULER}")
print(f"  adam       : beta1={ADAM_BETA1}, beta2={ADAM_BETA2}, eps={ADAM_EPSILON}")
print(f"  seq_len    : {MAX_SEQ_LEN}")


## Setup & Model Loading

In [2]:
import os, glob, sys, subprocess, site

candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)

if not candidates:
    raise FileNotFoundError("No Triton wheel found under /kaggle/input")
wheel = candidates[0]

target = "/kaggle/working/pydeps"
os.makedirs(target, exist_ok=True)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-deps",
        "--target", target,
        "--upgrade",
        "--ignore-installed",
        wheel,
    ],
    check=True,
)

if target not in sys.path:
    sys.path.insert(0, target)

site.addsitedir(target)

print("Custom target added:", target)

import importlib.util
print("triton spec：", importlib.util.find_spec("triton"))


Found Triton wheels: ['/kaggle/input/datasets/mayukh18/nemotron-packages/packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl', '/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl', '/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/triton-3.5.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl']
Processing /kaggle/input/datasets/mayukh18/nemotron-packages/packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
Custom target added: /kaggle/working/pydeps
triton spec： ModuleSpec(name='triton', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7ee63cc88620>, origin='/kaggle/working/pydeps/triton/__init__.py', submodule_search_locations=['/kaggle/working/pydeps/triton'])


In [ ]:

import sys, os, shutil, stat

sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')

ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
ptxas_dst = '/tmp/ptxas-blackwell'
if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
    shutil.copy2(ptxas_src, ptxas_dst)
    os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    src_bin = os.path.dirname(ptxas_src)
    dst_bin = '/tmp/triton_nvidia_bin'
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst

    import triton.backends.nvidia as nv_backend
    nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
    os.environ['TRITON_PTXAS_PATH'] = ptxas_dst

import triton.backends.nvidia.compiler as nv_compiler
nv_compiler.get_ptxas_version = lambda arch: '12.0'

print('Training environment fixes applied.')


In [ ]:

import glob, os, subprocess, sys

def recursive_wheels(pattern: str):
    return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

packages_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
all_mamba  = recursive_wheels("mamba_ssm-*.whl")
all_causal = recursive_wheels("causal*conv1d*.whl")

import torch
print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not os.path.isdir(packages_dir):
    raise FileNotFoundError(f"Offline wheel directory not found: {packages_dir}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "--no-index", "--find-links", packages_dir,
     "unsloth", "trl", "peft", "transformers", "datasets", "accelerate", "bitsandbytes"],
    check=True,
)

causal_wheel = all_causal[-1] if all_causal else None
mamba_wheel  = all_mamba[-1]  if all_mamba  else None

if causal_wheel:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", causal_wheel], check=True)
if mamba_wheel:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", mamba_wheel], check=True)
else:
    raise FileNotFoundError("Could not find a compatible mamba_ssm wheel.")

print("Offline package installation finished.")


In [ ]:

import torch
import kagglehub
from unsloth import FastLanguageModel

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
print(f"Model path: {MODEL_PATH}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=False,
    load_in_8bit=False,
    full_finetuning=False,
    trust_remote_code=True,
    unsloth_force_compile=False,
    attn_implementation="eager",
    dtype=torch.bfloat16,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Model loaded.")


In [ ]:

from unsloth import FastLanguageModel

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
model.print_trainable_parameters()


## Training

In [ ]:

import pandas as pd
import random
import re, gc, time, math
from collections import defaultdict
from datasets import Dataset as HFDataset
from trl import SFTTrainer, SFTConfig
from torch.utils.data import DataLoader, Sampler

df = pd.read_csv(DATASET_PATH)
print(f"Loaded: {len(df):,} rows")
print(f"Columns: {list(df.columns)}")
print(f"Unique types: {sorted(df['type'].unique())}")

# ── Sample per type ────────────────────────────────────────────────────────
sampled_dfs = []
for type_name, n in TYPE_SAMPLES.items():
    subset = df[df["type"] == type_name]
    if len(subset) == 0:
        print(f"  WARNING: no rows for '{type_name}' — skipping")
        continue
    if n is None or n >= len(subset):
        sampled_dfs.append(subset)
        print(f"  {type_name:<28} {len(subset):>5} / {len(subset)} (all)")
    else:
        sampled_dfs.append(subset.sample(n, random_state=SEED))
        print(f"  {type_name:<28} {n:>5} / {len(subset)}")

df_sampled = pd.concat(sampled_dfs).sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f"\nTotal training rows: {len(df_sampled):,}")

# ── Build messages from prompt / generated_cot / answer ───────────────────
records      = []
record_types = []
for _, row in df_sampled.iterrows():
    prompt = str(row["prompt"])
    answer = str(row["answer"])
    cot    = str(row["generated_cot"])
    if not cot or cot == "nan" or len(cot.strip()) < 5:
        continue
    cot_cleaned = re.sub(r'\\boxed\{[^}]*\}', '', cot).rstrip()
    user_content      = prompt + PROMPT_SUFFIX
    assistant_content = cot_cleaned + f"\n</think>\n\\boxed{{{answer}}}"
    records.append({"messages": [
        {"role": "user",      "content": user_content},
        {"role": "assistant", "content": assistant_content},
    ]})
    record_types.append(str(row["type"]))

dataset = HFDataset.from_list(records)
print(f"SFT records: {len(records):,}")


def formatting_prompts_func(example):
    messages = example["messages"]
    if messages and isinstance(messages[0], dict):
        conversations = [messages]
    else:
        conversations = messages
    texts = []
    for conversation in conversations:
        try:
            text = tokenizer.apply_chat_template(
                conversation, tokenize=False,
                add_generation_prompt=False, enable_thinking=True,
            )
        except TypeError:
            text = tokenizer.apply_chat_template(
                conversation, tokenize=False, add_generation_prompt=False,
            )
        texts.append(text)
    return texts


training_args = SFTConfig(
    output_dir=f"{OUTPUT_DIR}/sft_output",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_steps=WARMUP_STEPS,
    max_length=MAX_SEQ_LEN,
    adam_beta1=ADAM_BETA1,
    adam_beta2=ADAM_BETA2,
    adam_epsilon=ADAM_EPSILON,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    logging_steps=LOGGING_STEPS,
    save_strategy="no",
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=DATALOADER_NUM_WORKERS,
    remove_unused_columns=False,
    seed=SEED,
    report_to="none",
    packing=False,
)


def build_stratified_index_order(labels, batch_size, seed):
    by_label = defaultdict(list)
    for idx, label in enumerate(labels):
        by_label[label].append(idx)
    rng = random.Random(seed)
    for idx_list in by_label.values():
        rng.shuffle(idx_list)
    n_batches = max(1, math.ceil(len(labels) / batch_size))
    batches = [[] for _ in range(n_batches)]
    batch_order = list(range(n_batches))
    rng.shuffle(batch_order)
    assigned = 0
    for label in sorted(by_label.keys()):
        for idx in by_label[label]:
            batches[batch_order[assigned % n_batches]].append(idx)
            assigned += 1
    return [idx for batch in batches for idx in batch]


class PrecomputedOrderSampler(Sampler):
    def __init__(self, order):
        self.order = list(order)
    def __iter__(self):
        return iter(self.order)
    def __len__(self):
        return len(self.order)


class StratifiedSFTTrainer(SFTTrainer):
    def __init__(self, *args, stratified_order=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.stratified_order = stratified_order

    def get_train_dataloader(self):
        if self.stratified_order is None:
            return super().get_train_dataloader()
        dataloader_kwargs = {
            "batch_size":         self.args.per_device_train_batch_size,
            "sampler":            PrecomputedOrderSampler(self.stratified_order),
            "collate_fn":         self.data_collator,
            "num_workers":        self.args.dataloader_num_workers,
            "pin_memory":         self.args.dataloader_pin_memory,
            "persistent_workers": self.args.dataloader_persistent_workers,
            "drop_last":          self.args.dataloader_drop_last,
        }
        if self.args.dataloader_num_workers > 0:
            dataloader_kwargs["prefetch_factor"] = self.args.dataloader_prefetch_factor
        return DataLoader(self.train_dataset, **dataloader_kwargs)


effective_batch = BATCH_SIZE * GRAD_ACCUM
stratified_order = build_stratified_index_order(record_types, effective_batch, SEED)
print(f"Effective batch size: {effective_batch}")

trainer = StratifiedSFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    formatting_func=formatting_prompts_func,
    stratified_order=stratified_order,
)

# ── Print GPU memory before training ──────────────────────────────────────
import torch
torch.cuda.reset_peak_memory_stats()
allocated = torch.cuda.memory_allocated() / 1024**3
reserved  = torch.cuda.memory_reserved()  / 1024**3
print(f"GPU memory before training: allocated={allocated:.2f} GB, reserved={reserved:.2f} GB")

print("Starting SFT training...")
t0 = time.time()
trainer.train()
elapsed = time.time() - t0
print(f"Training done in {elapsed/60:.1f} min")

# ── Print peak GPU memory after training ──────────────────────────────────
peak_mem = torch.cuda.max_memory_allocated() / 1024**3
print(f"Peak GPU memory during training: {peak_mem:.2f} GB / 96 GB ({peak_mem/96*100:.1f}% utilization)")

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to {ADAPTER_DIR}")


## Create submission.zip

In [ ]:

import json, os, shutil, zipfile

OUTPUT_DIR            = "/kaggle/working"
SUBMISSION_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "submission_adapter")
os.makedirs(SUBMISSION_ADAPTER_DIR, exist_ok=True)

src_adapter_dir = "/kaggle/working/sft_adapter"
required_files  = ["adapter_config.json", "adapter_model.safetensors"]

for fname in required_files:
    src = os.path.join(src_adapter_dir, fname)
    dst = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
    if not os.path.exists(src):
        raise FileNotFoundError(f"Missing: {src}")
    shutil.copy2(src, dst)
    print(f"Copied {fname} ({os.path.getsize(dst)/1024/1024:.1f} MB)")

# Set inference config
config_path = os.path.join(SUBMISSION_ADAPTER_DIR, "adapter_config.json")
with open(config_path) as f:
    cfg = json.load(f)
cfg["base_model_name_or_path"] = BASE_MODEL_NAME
cfg["inference_mode"] = True
cfg["lora_dropout"]   = 0.0
with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

zip_path = os.path.join(OUTPUT_DIR, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        fpath = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
        zf.write(fpath, fname)
        print(f"  Added {fname}")

print(f"\nsubmission.zip: {os.path.getsize(zip_path)/1024/1024:.1f} MB")
print("Done! Ready to submit.")
